# LC-DA-CA1 schematic model

Author: Dinghao Luo  
Date  : 26 March 2026

This notebook distills the exploratory modelling (see `general_model_EDA.ipynb`) into one stabilised parameter set and evaluates it with `20` paired synthetic-population replicates.

Each replicate does two things:
- draws a fresh CA1 population from the same parameter distributions
- compares baseline, LC activation, and DA blockade on that **same** synthetic population

The three figure blocks below mirror the manuscript-style experimental comparisons:
1. baseline vs **1.5x LC activation** conditions (Fig. 2)
2. DA-targeted vs non-targeted CA1 cells (Fig. 5, top)
3. baseline vs **DA-blocked** conditions (Fig. 5, bottom)

This notebook implements a simple, schematic model of the LC-DA-CA1 PyrUp dataset from Luo et al. (2026). The goal is not to build a detailed biophysical model, but to support a mechanistic interpretation of the main results in the paper.

In this version, CA1 pyramidal neurones are driven by two direct task-related signals and a delayed dopamine release signal:
- $R(t)$: a run-related drive that rises after run onset and then decays
- $W(t)$: a reward-associated suppressive drive that dips after run onset and then recovers
- $D_{\mathrm{dip}}(t)$: a pre-run DA dip that is shown in the DA signal plots for context
- $D_{\mathrm{rel}}(t)$: a delayed LC-driven positive DA release signal

For each CA1 cell $i$, the baseline input is
$$x_i(t) = b_i + w_i^R R(t) + w_i^W W(t)$$

This is converted into a non-negative firing-rate-like quantity using a softplus function ($\phi$ here):
$$r_i^0(t) = \phi(x_i(t))$$

The positive LC-driven DA component is then gated smoothly by the current pre-DA firing rate:
$$m_i(t) = \sigma\left(\frac{r_i^0(t) - r_{1/2}}{k_r}\right)$$
where $r_{1/2}$ sets the turning point of the DA effect and $k_r$ controls how sharply that effect ramps up. With this form, DA effects stay small below the chosen turning point and grow smoothly above it, without a hard threshold.

The key modelling assumption is that the **positive LC-driven DA component** is what modulates CA1 gain. The pre-run DA dip is still shown in the DA signal traces, but it does not directly modulate CA1 firing in this simplified notebook. That avoids an implausible selective pre-run CA1 effect, because before run onset the model is not meant to push large numbers of cells above the DA-sensitive regime.

The DA-scaled drive is
$$r_i^{\mathrm{drive}}(t) = r_i^0(t)\left[1 + g_i D_{\mathrm{rel}}(t)m_i(t)\right]$$

Finally, each cell has its own intrinsic recovery time constant. In the implementation below, rising drive is followed quickly, but when the drive starts to fall the cell relaxes back towards baseline with a cell-specific time constant $\tau_i^{\mathrm{intr}}$. The direct run and reward drives therefore return relatively early, and the final tail of each cell's activity profile is shaped mainly by this intrinsic recovery.

Here:
- $b_i$ is the baseline excitability of cell $i$
- $w_i^R$ and $w_i^W$ are that cell's sensitivities to run and reward-related inputs
- $r_{1/2}$ is the DA-gating turning point in firing-rate units
- $k_r$ controls how gradually targeted positive DA modulation turns on
- $g_i$ sets the strength of the targeted positive DA gain modulation
- $\tau_i^{\mathrm{intr}}$ is the cell-specific intrinsic recovery time constant


In [1]:
# imports
import numpy as np
import matplotlib.pyplot as plt

from copy import deepcopy
from dataclasses import dataclass
from scipy.stats import ttest_rel, wilcoxon

In [2]:
@dataclass
class PARAMS:
    # simulation grid
    dt: float = 0.01
    t_pre: float = 1.00
    t_post: float = 6.00

    # bootstrap controls
    n_bootstrap: int = 20
    seed_start: int = 0
    lc_activation_fold: float = 1.50

    # population size
    n_cells: int = 1000

    # population priors
    baseline_mean: float = 1.80
    baseline_sd: float = 0.35

    wR_mean: float = 1.00
    wR_sd: float = 0.75
    wW_mean: float = 0.60
    wW_sd: float = 0.90

    # DA-targeting and smooth rate dependence
    frac_da_targ: float = 0.35
    da_half_rate: float = 2.50
    da_rate_slope: float = 0.35
    da_gain: float = 0.80

    # cell-intrinsic recovery constants
    intrinsic_tau_mean: float = 0.80
    intrinsic_tau_sd: float = 0.18
    intrinsic_tau_min: float = 0.35
    intrinsic_tau_max: float = 1.40

    # static nonlinearity / output limits
    softplus_beta: float = 2.00
    max_rate: float = 20.00

    # drive timings
    drive_end: float = 5.00
    direct_drive_end: float = 3.00
    run_peak_t: float = 1.00
    reward_trough_t: float = 1.00
    da_dip_t: float = -0.25
    da_dip_tau: float = 0.20

    # drive amplitudes
    run_amp: float = 1.10
    reward_amp: float = 1.10
    da_dip_amp: float = -0.50

    # LC drive shape
    lc_baseline: float = 1.00
    lc_amp: float = 1.50
    lc_mu: float = 0.00
    lc_sigma: float = 0.25

    # LC -> DA conversion
    lc_to_da_gain: float = 1.60
    da_tau_decay: float = 0.90

    # population analysis
    pre_window: tuple = (-1.00, -0.50)
    post_window: tuple = (0.50, 1.50)
    up_thresh: float = 3 / 2
    down_thresh: float = 2 / 3

    # numerical safeguard
    eps: float = 1e-6


class_colors = {
    'is_up': 'darkorange',
    'is_other': 'grey',
    'is_down': 'purple'
}

condition_colors = {
    'baseline': '0.35',
    'lc': class_colors['is_up'],
    'blocked': class_colors['is_down'],
    'da_targeted': class_colors['is_up'],
    'not_targeted': '0.45'
}


In [3]:
# helper functions

def window_mask(t, window):
    return (t >= window[0]) & (t < window[1])


def response_strength(rates, t, p):
    pre_mask = window_mask(t, p.pre_window)
    post_mask = window_mask(t, p.post_window)

    pre_mean = np.mean(rates[:, pre_mask], axis=1)
    post_mean = np.mean(rates[:, post_mask], axis=1)
    return post_mean / (pre_mean + p.eps)


def classify_cells(resp, p):
    is_up = resp >= p.up_thresh
    is_down = resp <= p.down_thresh
    is_other = ~(is_up | is_down)
    return {
        'is_up': is_up,
        'is_down': is_down,
        'is_other': is_other,
    }


def safe_mean_trace(rate_matrix, mask):
    if np.sum(mask) == 0:
        return np.full(rate_matrix.shape[1], np.nan)
    return np.mean(rate_matrix[mask], axis=0)


def sem(values, axis=0):
    values = np.asarray(values, dtype=float)
    count = np.sum(np.isfinite(values), axis=axis)
    spread = np.nanstd(values, axis=axis, ddof=1)
    out = spread / np.sqrt(np.maximum(count, 1))
    if np.isscalar(out):
        return np.nan if count <= 1 else float(out)
    out = np.asarray(out, dtype=float)
    out = np.where(count > 1, out, np.nan)
    return out


def describe_scalar(values):
    values = np.asarray(values, dtype=float)
    return float(np.nanmean(values)), float(sem(values, axis=0))


def baseline_subtracted_traces(traces, t, window):
    traces = np.asarray(traces, dtype=float)
    baseline = np.mean(traces[:, window_mask(t, window)], axis=1, keepdims=True)
    return traces - baseline


def format_p_value(p):
    if not np.isfinite(p):
        return 'nan'
    if p < 1e-4:
        return '<1e-4'
    return f'{p:.4f}'


def paired_test_summary(left, right):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    mask = np.isfinite(left) & np.isfinite(right)
    left = left[mask]
    right = right[mask]

    if len(left) == 0:
        return {
            'n': 0,
            'mean_left': np.nan,
            'mean_right': np.nan,
            'mean_diff': np.nan,
            'ttest_stat': np.nan,
            'ttest_p': np.nan,
            'wilcoxon_stat': np.nan,
            'wilcoxon_p': np.nan,
        }

    t_res = ttest_rel(right, left, nan_policy='omit')
    try:
        w_res = wilcoxon(right, left, zero_method='wilcox', method='auto')
        w_stat = float(w_res.statistic)
        w_p = float(w_res.pvalue)
    except ValueError:
        w_stat = np.nan
        w_p = np.nan

    return {
        'n': int(len(left)),
        'mean_left': float(np.mean(left)),
        'mean_right': float(np.mean(right)),
        'mean_diff': float(np.mean(right - left)),
        'ttest_stat': float(t_res.statistic),
        'ttest_p': float(t_res.pvalue),
        'wilcoxon_stat': w_stat,
        'wilcoxon_p': w_p,
    }


def print_paired_summary(label, left, right):
    summary = paired_test_summary(left, right)
    print(
        f'{label}: '
        f'delta = {summary["mean_diff"]:.2f}, '
        f'paired t p = {format_p_value(summary["ttest_p"])}, '
        f'Wilcoxon p = {format_p_value(summary["wilcoxon_p"])}'
    )


def plot_mean_sem(ax, t, traces, color, label, linestyle='-', alpha_fill=0.18):
    traces = np.asarray(traces, dtype=float)
    mean_trace = np.nanmean(traces, axis=0)
    sem_trace = sem(traces, axis=0)
    ax.plot(t, mean_trace, color=color, linewidth=2, linestyle=linestyle, label=label)
    ax.fill_between(t, mean_trace - sem_trace, mean_trace + sem_trace, color=color, alpha=alpha_fill, linewidth=0)


def plot_paired_bar_scatter(ax, left, right, labels, colors, ylabel, title):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    mask = np.isfinite(left) & np.isfinite(right)
    left = left[mask]
    right = right[mask]

    order = np.argsort((left + right) / 2.0)
    left = left[order]
    right = right[order]

    means = np.array([np.mean(left), np.mean(right)], dtype=float)
    sems = np.array([sem(left, axis=0), sem(right, axis=0)], dtype=float)
    x = np.array([0.0, 1.0], dtype=float)

    ax.bar(x, means, width=0.56, color=colors, alpha=0.55, edgecolor='none', zorder=0)
    ax.errorbar(x, means, yerr=sems, fmt='none', ecolor='k', elinewidth=1.2, capsize=4, capthick=1.2, zorder=3)

    x_left = np.full(len(left), x[0], dtype=float)
    x_right = np.full(len(right), x[1], dtype=float)

    for xl, xr, yl, yr in zip(x_left, x_right, left, right):
        ax.plot([xl, xr], [yl, yr], color='0.75', linewidth=0.8, zorder=1)

    ax.scatter(x_left, left, s=26, color=colors[0], edgecolor='white', linewidth=0.4, zorder=2)
    ax.scatter(x_right, right, s=26, color=colors[1], edgecolor='white', linewidth=0.4, zorder=2)

    y_all = np.concatenate([left, right, means])
    y_span = float(np.nanmax(y_all) - np.nanmin(y_all))
    if y_span <= 0:
        y_span = max(abs(float(np.nanmean(y_all))) * 0.10, 0.10)
    y_pad = 0.06 * y_span

    for xi, mean_i, sem_i in zip(x, means, sems):
        ax.text(xi, mean_i + sem_i + y_pad, f'{mean_i:.2f}\n+/-{sem_i:.2f} SEM', ha='center', va='bottom', fontsize=8)

    summary = paired_test_summary(left, right)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.20)

    stats_text = (
        f'delta = {summary["mean_diff"]:.2f}\n'
        f'paired t p = {format_p_value(summary["ttest_p"])}\n'
        f'Wilcoxon p = {format_p_value(summary["wilcoxon_p"])}'
    )
    ax.text(
        0.02,
        0.98,
        stats_text,
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=9,
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='0.8', alpha=0.90),
    )
    return summary


def half_cosine(tt, t0, t1, y0, y1):
    s = np.clip((tt - t0) / (t1 - t0), 0.0, 1.0)
    return y0 + (y1 - y0) * 0.5 * (1.0 - np.cos(np.pi * s))


def exp_kernel(dt, tau, t_max, eps):
    tk = np.arange(0.0, t_max + dt, dt)
    k = np.exp(-tk / tau)
    k /= (np.sum(k) * dt + eps)
    return tk, k


def make_drives(t, p):
    R = np.zeros_like(t, dtype=float)
    W = np.zeros_like(t, dtype=float)
    D_dip = np.zeros_like(t, dtype=float)

    rise_mask = (t >= 0.0) & (t <= p.run_peak_t)
    decay_mask = (t > p.run_peak_t) & (t <= p.direct_drive_end)
    R[rise_mask] = half_cosine(t[rise_mask], 0.0, p.run_peak_t, 0.0, p.run_amp)
    R[decay_mask] = half_cosine(t[decay_mask], p.run_peak_t, p.direct_drive_end, p.run_amp, 0.0)

    down_mask = (t >= 0.0) & (t <= p.reward_trough_t)
    recover_mask = (t > p.reward_trough_t) & (t <= p.direct_drive_end)
    W[down_mask] = half_cosine(t[down_mask], 0.0, p.reward_trough_t, 0.0, -p.reward_amp)
    W[recover_mask] = half_cosine(t[recover_mask], p.reward_trough_t, p.direct_drive_end, -p.reward_amp, 0.0)

    dip_down_mask = (t >= -p.t_pre) & (t <= p.da_dip_t)
    dip_recover_mask = t > p.da_dip_t
    D_dip[dip_down_mask] = half_cosine(t[dip_down_mask], -p.t_pre, p.da_dip_t, 0.0, p.da_dip_amp)
    D_dip[dip_recover_mask] = p.da_dip_amp * np.exp(-(t[dip_recover_mask] - p.da_dip_t) / p.da_dip_tau)

    L = p.lc_baseline + p.lc_amp * np.exp(-0.5 * ((t - p.lc_mu) / p.lc_sigma) ** 2)
    lc_excess = np.maximum(L - p.lc_baseline, 0.0)

    _, k = exp_kernel(p.dt, p.da_tau_decay, p.t_pre + p.t_post, p.eps)
    D_release = p.lc_to_da_gain * np.convolve(lc_excess, k, mode='full')[: len(t)] * p.dt
    D = D_dip + D_release

    return {
        'R': R,
        'W': W,
        'D': D,
        'L': L,
        'D_dip': D_dip,
        'D_release': D_release,
        'lc_excess': lc_excess,
    }


In [4]:
# simulation functions

def apply_intrinsic_decay_matrix(r_drive, tau_intr, dt, max_rate):
    r = np.zeros_like(r_drive)
    r[:, 0] = r_drive[:, 0]
    alpha = dt / np.maximum(tau_intr, 1e-6)

    for k in range(1, r_drive.shape[1]):
        target = r_drive[:, k]
        prev = r[:, k - 1]
        rising = target >= prev

        r[rising, k] = target[rising]
        r[~rising, k] = prev[~rising] + alpha[~rising] * (target[~rising] - prev[~rising])

    return np.clip(r, 0.0, max_rate)


def make_population(p, rng):
    da_u = rng.random(p.n_cells)
    return {
        'b': rng.normal(p.baseline_mean, p.baseline_sd, p.n_cells),
        'wR': rng.normal(p.wR_mean, p.wR_sd, p.n_cells),
        'wW': rng.normal(p.wW_mean, p.wW_sd, p.n_cells),
        'tau_intr': np.clip(
            rng.normal(p.intrinsic_tau_mean, p.intrinsic_tau_sd, p.n_cells),
            p.intrinsic_tau_min,
            p.intrinsic_tau_max,
        ),
        'da_u': da_u,
        'da_targ': da_u < p.frac_da_targ,
    }


def simulate_population_condition(t, p, pop):
    drives = make_drives(t, p)
    R = drives['R']
    W = drives['W']

    x = (
        pop['b'][:, None]
        + pop['wR'][:, None] * R[None, :]
        + pop['wW'][:, None] * W[None, :]
    )

    xb = p.softplus_beta * x
    r0 = (np.log1p(np.exp(-np.abs(xb))) + np.maximum(xb, 0.0)) / p.softplus_beta

    slope = max(p.da_rate_slope, p.eps)
    m = 1.0 / (1.0 + np.exp(-(r0 - p.da_half_rate) / slope))

    targeted_gain = 1.0 + pop['da_targ'][:, None].astype(float) * p.da_gain * m * drives['D_release'][None, :]
    r_drive = np.clip(r0 * targeted_gain, 0.0, p.max_rate)
    rates = apply_intrinsic_decay_matrix(r_drive, pop['tau_intr'], p.dt, p.max_rate)

    resp = response_strength(rates, t, p)
    classes = classify_cells(resp, p)

    return {
        't': t,
        'drives': drives,
        'r0': r0,
        'm': m,
        'r_drive': r_drive,
        'rates': rates,
        'resp': resp,
        'classes': classes,
        'pre_mask': window_mask(t, p.pre_window),
        'post_mask': window_mask(t, p.post_window),
        'mean_traces': {
            'is_up': safe_mean_trace(rates, classes['is_up']),
            'is_other': safe_mean_trace(rates, classes['is_other']),
            'is_down': safe_mean_trace(rates, classes['is_down']),
        },
    }


def run_bootstrap_suite(p):
    t = np.arange(-p.t_pre, p.t_post, p.dt)

    p_lc = deepcopy(p)
    p_lc.lc_amp = p.lc_amp * p.lc_activation_fold

    p_block = deepcopy(p)
    p_block.lc_to_da_gain = 0.0

    base_up_traces = []
    lc_up_traces = []
    block_up_traces = []
    base_down_traces = []
    lc_down_traces = []
    block_down_traces = []
    targeted_traces = []
    non_targeted_traces = []

    stats = {
        'base_up_pct': [],
        'base_down_pct': [],
        'lc_up_pct': [],
        'lc_down_pct': [],
        'block_up_pct': [],
        'block_down_pct': [],
        'p_up_da_targeted': [],
        'p_up_not_targeted': [],
        'post_rate_da_targeted': [],
        'post_rate_not_targeted': [],
    }

    for seed in range(p.seed_start, p.seed_start + p.n_bootstrap):
        rng = np.random.default_rng(seed)
        pop = make_population(p, rng)

        base = simulate_population_condition(t, p, pop)
        lc = simulate_population_condition(t, p_lc, pop)
        block = simulate_population_condition(t, p_block, pop)

        base_up_traces.append(base['mean_traces']['is_up'])
        lc_up_traces.append(lc['mean_traces']['is_up'])
        block_up_traces.append(block['mean_traces']['is_up'])

        base_down_traces.append(base['mean_traces']['is_down'])
        lc_down_traces.append(lc['mean_traces']['is_down'])
        block_down_traces.append(block['mean_traces']['is_down'])

        da_mask = pop['da_targ']
        not_da_mask = ~da_mask
        targeted_traces.append(safe_mean_trace(base['rates'], da_mask))
        non_targeted_traces.append(safe_mean_trace(base['rates'], not_da_mask))

        stats['base_up_pct'].append(100 * np.mean(base['classes']['is_up']))
        stats['base_down_pct'].append(100 * np.mean(base['classes']['is_down']))
        stats['lc_up_pct'].append(100 * np.mean(lc['classes']['is_up']))
        stats['lc_down_pct'].append(100 * np.mean(lc['classes']['is_down']))
        stats['block_up_pct'].append(100 * np.mean(block['classes']['is_up']))
        stats['block_down_pct'].append(100 * np.mean(block['classes']['is_down']))
        stats['p_up_da_targeted'].append(100 * np.mean(base['classes']['is_up'][da_mask]))
        stats['p_up_not_targeted'].append(100 * np.mean(base['classes']['is_up'][not_da_mask]))
        stats['post_rate_da_targeted'].append(np.mean(base['rates'][da_mask][:, base['post_mask']]))
        stats['post_rate_not_targeted'].append(np.mean(base['rates'][not_da_mask][:, base['post_mask']]))

    stats = {key: np.asarray(value, dtype=float) for key, value in stats.items()}

    return {
        't': t,
        'params': p,
        'base_drives': make_drives(t, p),
        'lc_drives': make_drives(t, p_lc),
        'block_drives': make_drives(t, p_block),
        'base_up_traces': np.asarray(base_up_traces, dtype=float),
        'lc_up_traces': np.asarray(lc_up_traces, dtype=float),
        'block_up_traces': np.asarray(block_up_traces, dtype=float),
        'base_down_traces': np.asarray(base_down_traces, dtype=float),
        'lc_down_traces': np.asarray(lc_down_traces, dtype=float),
        'block_down_traces': np.asarray(block_down_traces, dtype=float),
        'targeted_traces': np.asarray(targeted_traces, dtype=float),
        'non_targeted_traces': np.asarray(non_targeted_traces, dtype=float),
        'stats': stats,
    }


In [5]:
# run the paired synthetic-population bootstrap suite
p = PARAMS()
results = run_bootstrap_suite(p)


def print_summary_line(label, values):
    mean_v, sem_v = describe_scalar(values)
    print(f'{label}: {mean_v:.2f} +/- {sem_v:.2f} SEM')


print('Bootstrap setup:')
print(f'  paired population replicates = {p.n_bootstrap}')
print(f'  cells per replicate = {p.n_cells}')
print(f'  LC activation fold = {p.lc_activation_fold:.2f}x')
print()

print('Key paired summaries (mean +/- SEM):')
print_summary_line('  baseline PyrUp (%)', results['stats']['base_up_pct'])
print_summary_line('  LC-activated PyrUp (%)', results['stats']['lc_up_pct'])
print_paired_summary('  LC activation effect on PyrUp (%)', results['stats']['base_up_pct'], results['stats']['lc_up_pct'])
print_summary_line('  baseline PyrDown (%)', results['stats']['base_down_pct'])
print_summary_line('  LC-activated PyrDown (%)', results['stats']['lc_down_pct'])
print_paired_summary('  LC activation effect on PyrDown (%)', results['stats']['base_down_pct'], results['stats']['lc_down_pct'])
print_summary_line('  DA-blocked PyrUp (%)', results['stats']['block_up_pct'])
print_paired_summary('  DA blockade effect on PyrUp (%)', results['stats']['base_up_pct'], results['stats']['block_up_pct'])
print_summary_line('  DA-blocked PyrDown (%)', results['stats']['block_down_pct'])
print_paired_summary('  DA blockade effect on PyrDown (%)', results['stats']['base_down_pct'], results['stats']['block_down_pct'])
print_summary_line('  P(PyrUp | DA-targeted) (%)', results['stats']['p_up_da_targeted'])
print_summary_line('  P(PyrUp | not targeted) (%)', results['stats']['p_up_not_targeted'])
print_paired_summary('  PyrUp enrichment in DA-targeted cells (%)', results['stats']['p_up_not_targeted'], results['stats']['p_up_da_targeted'])
print_summary_line('  post-run rate, DA-targeted (Hz)', results['stats']['post_rate_da_targeted'])
print_summary_line('  post-run rate, not targeted (Hz)', results['stats']['post_rate_not_targeted'])
print_paired_summary('  Post-run rate enrichment in DA-targeted cells (Hz)', results['stats']['post_rate_not_targeted'], results['stats']['post_rate_da_targeted'])

Bootstrap setup:
  paired population replicates = 20
  cells per replicate = 1000
  LC activation fold = 1.50x

Key paired summaries (mean +/- SEM):
  baseline PyrUp (%): 37.48 +/- 0.37 SEM
  LC-activated PyrUp (%): 38.70 +/- 0.36 SEM
  LC activation effect on PyrUp (%): delta = 1.22, paired t p = <1e-4, Wilcoxon p = <1e-4
  baseline PyrDown (%): 5.55 +/- 0.13 SEM
  LC-activated PyrDown (%): 5.40 +/- 0.14 SEM
  LC activation effect on PyrDown (%): delta = -0.16, paired t p = <1e-4, Wilcoxon p = 0.0003
  DA-blocked PyrUp (%): 32.13 +/- 0.33 SEM
  DA blockade effect on PyrUp (%): delta = -5.35, paired t p = <1e-4, Wilcoxon p = <1e-4
  DA-blocked PyrDown (%): 6.12 +/- 0.12 SEM
  DA blockade effect on PyrDown (%): delta = 0.57, paired t p = <1e-4, Wilcoxon p = <1e-4
  P(PyrUp | DA-targeted) (%): 47.11 +/- 0.63 SEM
  P(PyrUp | not targeted) (%): 32.35 +/- 0.43 SEM
  PyrUp enrichment in DA-targeted cells (%): delta = 14.76, paired t p = <1e-4, Wilcoxon p = <1e-4
  post-run rate, DA-targeted 

C:\Users\luod\AppData\Roaming\Python\Python39\site-packages\scipy\stats\_morestats.py:3255: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "


## 1. LC Activation

Within each bootstrap replicate, LC activation is applied as a **1.5-fold increase in LC amplitude** on the same synthetic population used for baseline. The panels below compare the deterministic LC and DA signals, the mean population traces, and paired bootstrap summaries shown as **bars with SEM** plus paired scatter points and connecting lines.

In [6]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)

base_drives = results['base_drives']
lc_drives = results['lc_drives']

axes[0, 0].plot(results['t'], base_drives['L'], color=condition_colors['baseline'], linewidth=2, label='baseline')
axes[0, 0].plot(results['t'], lc_drives['L'], color=condition_colors['lc'], linewidth=2, linestyle='--', label=f'{p.lc_activation_fold:.1f}x LC')
axes[0, 0].axvline(0, linestyle='--', color='k', linewidth=1)
axes[0, 0].set_xlim([-1.0, 4.0])
axes[0, 0].set_xlabel('time from run onset (s)')
axes[0, 0].set_ylabel('LC signal (a.u.)')
axes[0, 0].set_title('LC activity')
axes[0, 0].legend(frameon=False)

axes[0, 1].plot(results['t'], base_drives['D'], color=condition_colors['baseline'], linewidth=2, label='baseline')
axes[0, 1].plot(results['t'], lc_drives['D'], color=condition_colors['lc'], linewidth=2, linestyle='--', label=f'{p.lc_activation_fold:.1f}x LC')
axes[0, 1].axvline(0, linestyle='--', color='k', linewidth=1)
axes[0, 1].set_xlim([-1.0, 4.0])
axes[0, 1].set_xlabel('time from run onset (s)')
axes[0, 1].set_ylabel('DA signal (a.u.)')
axes[0, 1].set_title('DA signal')
axes[0, 1].legend(frameon=False)

plot_paired_bar_scatter(
    axes[0, 2],
    results['stats']['base_up_pct'],
    results['stats']['lc_up_pct'],
    labels=['baseline', f'{p.lc_activation_fold:.1f}x LC'],
    colors=[condition_colors['baseline'], condition_colors['lc']],
    ylabel='PyrUp fraction (%)',
    title='Paired PyrUp fraction',
)

plot_mean_sem(axes[1, 0], results['t'], results['base_up_traces'], condition_colors['baseline'], 'baseline')
plot_mean_sem(axes[1, 0], results['t'], results['lc_up_traces'], condition_colors['lc'], f'{p.lc_activation_fold:.1f}x LC', linestyle='--')
axes[1, 0].axvline(0, linestyle='--', color='k', linewidth=1)
axes[1, 0].set_xlim([-1.0, 4.0])
axes[1, 0].set_xlabel('time from run onset (s)')
axes[1, 0].set_ylabel('mean rate (Hz)')
axes[1, 0].set_title('PyrUp population mean')
axes[1, 0].legend(frameon=False)

plot_mean_sem(axes[1, 1], results['t'], results['base_down_traces'], condition_colors['baseline'], 'baseline')
plot_mean_sem(axes[1, 1], results['t'], results['lc_down_traces'], condition_colors['lc'], f'{p.lc_activation_fold:.1f}x LC', linestyle='--')
axes[1, 1].axvline(0, linestyle='--', color='k', linewidth=1)
axes[1, 1].set_xlim([-1.0, 4.0])
axes[1, 1].set_xlabel('time from run onset (s)')
axes[1, 1].set_ylabel('mean rate (Hz)')
axes[1, 1].set_title('PyrDown population mean')
axes[1, 1].legend(frameon=False)

plot_paired_bar_scatter(
    axes[1, 2],
    results['stats']['base_down_pct'],
    results['stats']['lc_down_pct'],
    labels=['baseline', f'{p.lc_activation_fold:.1f}x LC'],
    colors=[condition_colors['baseline'], condition_colors['lc']],
    ylabel='PyrDown fraction (%)',
    title='Paired PyrDown fraction',
)

plt.show()

print('LC activation summary (mean +/- SEM):')
print_summary_line('  baseline PyrUp (%)', results['stats']['base_up_pct'])
print_summary_line(f'  {p.lc_activation_fold:.1f}x LC PyrUp (%)', results['stats']['lc_up_pct'])
print_paired_summary('  PyrUp fraction shift (%)', results['stats']['base_up_pct'], results['stats']['lc_up_pct'])
print_summary_line('  baseline PyrDown (%)', results['stats']['base_down_pct'])
print_summary_line(f'  {p.lc_activation_fold:.1f}x LC PyrDown (%)', results['stats']['lc_down_pct'])
print_paired_summary('  PyrDown fraction shift (%)', results['stats']['base_down_pct'], results['stats']['lc_down_pct'])

LC activation summary (mean +/- SEM):
  baseline PyrUp (%): 37.48 +/- 0.37 SEM
  1.5x LC PyrUp (%): 38.70 +/- 0.36 SEM
  PyrUp fraction shift (%): delta = 1.22, paired t p = <1e-4, Wilcoxon p = <1e-4
  baseline PyrDown (%): 5.55 +/- 0.13 SEM
  1.5x LC PyrDown (%): 5.40 +/- 0.14 SEM
  PyrDown fraction shift (%): delta = -0.16, paired t p = <1e-4, Wilcoxon p = 0.0003


C:\Users\luod\AppData\Local\Temp\ipykernel_121768\3877950751.py:62: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 2. DA-Targeted CA1 Cells Are More Likely PyrUp and More Active

This panel uses the baseline condition only. The targeted comparison is shown with a baseline-subtracted activity trace and paired bootstrap summary plots. The time trace is shown on the same `-1` to `4` s axis, but the key comparison is the **post-run** divergence rather than any selective pre-run modulation.

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)

targeted_delta = baseline_subtracted_traces(results['targeted_traces'], results['t'], p.pre_window)
non_targeted_delta = baseline_subtracted_traces(results['non_targeted_traces'], results['t'], p.pre_window)

plot_mean_sem(axes[0], results['t'], targeted_delta, condition_colors['da_targeted'], 'DA-targeted')
plot_mean_sem(axes[0], results['t'], non_targeted_delta, condition_colors['not_targeted'], 'not targeted')
axes[0].axvline(0, linestyle='--', color='k', linewidth=1)
axes[0].set_xlim([-1.0, 4.0])
axes[0].set_xlabel('time from run onset (s)')
axes[0].set_ylabel('delta rate from pre-window (Hz)')
axes[0].set_title('Activity by DA targeting')
axes[0].legend(frameon=False)

plot_paired_bar_scatter(
    axes[1],
    results['stats']['p_up_not_targeted'],
    results['stats']['p_up_da_targeted'],
    labels=['not targeted', 'DA-targeted'],
    colors=[condition_colors['not_targeted'], condition_colors['da_targeted']],
    ylabel='PyrUp fraction (%)',
    title='Paired PyrUp enrichment',
)

plot_paired_bar_scatter(
    axes[2],
    results['stats']['post_rate_not_targeted'],
    results['stats']['post_rate_da_targeted'],
    labels=['not targeted', 'DA-targeted'],
    colors=[condition_colors['not_targeted'], condition_colors['da_targeted']],
    ylabel='post-run mean rate (Hz)',
    title='Paired post-run activity',
)

plt.show()

print('DA-targeting summary (mean +/- SEM):')
print_summary_line('  P(PyrUp | DA-targeted) (%)', results['stats']['p_up_da_targeted'])
print_summary_line('  P(PyrUp | not targeted) (%)', results['stats']['p_up_not_targeted'])
print_paired_summary('  PyrUp enrichment in DA-targeted cells (%)', results['stats']['p_up_not_targeted'], results['stats']['p_up_da_targeted'])
print_summary_line('  post-run rate, DA-targeted (Hz)', results['stats']['post_rate_da_targeted'])
print_summary_line('  post-run rate, not targeted (Hz)', results['stats']['post_rate_not_targeted'])
print_paired_summary('  Post-run rate enrichment in DA-targeted cells (Hz)', results['stats']['post_rate_not_targeted'], results['stats']['post_rate_da_targeted'])

DA-targeting summary (mean +/- SEM):
  P(PyrUp | DA-targeted) (%): 47.11 +/- 0.63 SEM
  P(PyrUp | not targeted) (%): 32.35 +/- 0.43 SEM
  PyrUp enrichment in DA-targeted cells (%): delta = 14.76, paired t p = <1e-4, Wilcoxon p = <1e-4
  post-run rate, DA-targeted (Hz): 3.08 +/- 0.02 SEM
  post-run rate, not targeted (Hz): 2.38 +/- 0.01 SEM
  Post-run rate enrichment in DA-targeted cells (Hz): delta = 0.70, paired t p = <1e-4, Wilcoxon p = <1e-4


C:\Users\luod\AppData\Local\Temp\ipykernel_121768\1939483064.py:35: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 3. DA Blockade in CA1

Within each bootstrap replicate, the DA blockade condition removes the **LC-driven positive DA release** by setting `lc_to_da_gain = 0`. The panels below compare baseline and DA-blocked mean traces together with paired bootstrap summaries shown as **bars with SEM** plus paired scatter points and connecting lines.

In [8]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

plot_mean_sem(axes[0, 0], results['t'], results['base_up_traces'], condition_colors['baseline'], 'baseline')
plot_mean_sem(axes[0, 0], results['t'], results['block_up_traces'], condition_colors['blocked'], 'DA blocked', linestyle='--')
axes[0, 0].axvline(0, linestyle='--', color='k', linewidth=1)
axes[0, 0].set_xlim([-1.0, 4.0])
axes[0, 0].set_xlabel('time from run onset (s)')
axes[0, 0].set_ylabel('mean rate (Hz)')
axes[0, 0].set_title('PyrUp population mean')
axes[0, 0].legend(frameon=False)

plot_mean_sem(axes[0, 1], results['t'], results['base_down_traces'], condition_colors['baseline'], 'baseline')
plot_mean_sem(axes[0, 1], results['t'], results['block_down_traces'], condition_colors['blocked'], 'DA blocked', linestyle='--')
axes[0, 1].axvline(0, linestyle='--', color='k', linewidth=1)
axes[0, 1].set_xlim([-1.0, 4.0])
axes[0, 1].set_xlabel('time from run onset (s)')
axes[0, 1].set_ylabel('mean rate (Hz)')
axes[0, 1].set_title('PyrDown population mean')
axes[0, 1].legend(frameon=False)

plot_paired_bar_scatter(
    axes[1, 0],
    results['stats']['base_up_pct'],
    results['stats']['block_up_pct'],
    labels=['baseline', 'DA blocked'],
    colors=[condition_colors['baseline'], condition_colors['blocked']],
    ylabel='PyrUp fraction (%)',
    title='Paired PyrUp fraction',
)

plot_paired_bar_scatter(
    axes[1, 1],
    results['stats']['base_down_pct'],
    results['stats']['block_down_pct'],
    labels=['baseline', 'DA blocked'],
    colors=[condition_colors['baseline'], condition_colors['blocked']],
    ylabel='PyrDown fraction (%)',
    title='Paired PyrDown fraction',
)

plt.show()

print('DA blockade summary (mean +/- SEM):')
print_summary_line('  baseline PyrUp (%)', results['stats']['base_up_pct'])
print_summary_line('  DA-blocked PyrUp (%)', results['stats']['block_up_pct'])
print_paired_summary('  PyrUp fraction shift (%)', results['stats']['base_up_pct'], results['stats']['block_up_pct'])
print_summary_line('  baseline PyrDown (%)', results['stats']['base_down_pct'])
print_summary_line('  DA-blocked PyrDown (%)', results['stats']['block_down_pct'])
print_paired_summary('  PyrDown fraction shift (%)', results['stats']['base_down_pct'], results['stats']['block_down_pct'])

DA blockade summary (mean +/- SEM):
  baseline PyrUp (%): 37.48 +/- 0.37 SEM
  DA-blocked PyrUp (%): 32.13 +/- 0.33 SEM
  PyrUp fraction shift (%): delta = -5.35, paired t p = <1e-4, Wilcoxon p = <1e-4
  baseline PyrDown (%): 5.55 +/- 0.13 SEM
  DA-blocked PyrDown (%): 6.12 +/- 0.12 SEM
  PyrDown fraction shift (%): delta = 0.57, paired t p = <1e-4, Wilcoxon p = <1e-4


C:\Users\luod\AppData\Local\Temp\ipykernel_121768\727840305.py:41: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Interpretation and Caveats

This notebook should still be read as a **phenomenological systems-neuroscience model**, not a biophysical circuit model.

What the bootstrap adds:
- it makes the figure-level conclusions less dependent on one convenient RNG seed
- it keeps each manipulation **paired** to its own baseline population, which is the fairest comparison for this model

What it does **not** add:
- it does not infer uncertainty from recorded experimental cells directly
- it does not model trial-to-trial variability, recurrent CA1 circuitry, or receptor-level DA biology

So the figures here should be interpreted as support for the **logic of the model assumptions** under repeated synthetic-population resampling, not as proof that the real circuit uses these exact equations.
